**Name:** *Muhammad Bilal*

**Department:** *Data Science*

**University:** *Ghulam Ishaq Khan Institute of Engineering and Sciences*

# **Decode Labs Internship**

I used a Python Notebook since that is the medium I am most familiar with in terms of working with datasets and the tasks that need to be performed in the projects.

### **Data Science Project 2:** Supervised Learning & Fraud Detection
**Objective:** Build a mathematically secure classification pipeline to identify fraudulent/default transactions in highly imbalanced loan data, utilizing SMOTE and strict ROC-AUC evaluation.

### **Task 1:** Data Loading & The Holy Grail of No Leakage
**Goal:** Collect or load a dataset and safety for no leakages.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
print("Loading loan dataset...")
df = pd.read_csv('loan_data_new.csv')

# Separate Features (X) and Target (y)
# 'Loan Status' is our target variable (y) for fraud/default detection
X = df.drop('Loan Status', axis=1)
y = df['Loan Status']

# The Train/Test Split: We split BEFORE any scaling or SMOTE to prevent data leakage into the test set.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Class imbalance in training set:\n{y_train.value_counts(normalize=True) * 100}")

Loading loan dataset...
Training set shape: (36000, 13)
Test set shape: (9000, 13)
Class imbalance in training set:
Loan Status
0    77.777778
1    22.222222
Name: proportion, dtype: float64


### **Task 2:** The Zero-Leakage Preprocessing Pipeline
**Goal:** We use `ColumnTransformer` to handle numerical and categorical data independently. Then, we construct an `imblearn.pipeline` to ensure SMOTE is only applied to the training folds during Cross-Validation.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SklearnPipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

# Defining column types
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
numerical_cols = X_train.select_dtypes(exclude=['object', 'category']).columns

# Sub-pipeline for numerical features (Impute -> Scale)
numeric_transformer = SklearnPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Sub-pipeline for categorical features (Impute -> OneHotEncode)
categorical_transformer = SklearnPipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a single Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Construct the Master Imbalanced Pipeline
# ORDER IS CRITICAL: Preprocess -> SMOTE -> Classifier
pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("Zero-Leakage Pipeline constructed successfully.")

Zero-Leakage Pipeline constructed successfully.


### **Task 3:** Holistic Hyperparameter Tuning
**Goal:** We optimize the model using GridSearchCV, evaluating performance exclusively on `roc_auc` to ensure the model actually learns to separate the minority class rather than just guessing the majority.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the hyperparameter grid
# We keep it focused to ensure reasonable execution time in Colab/Jupyter
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

print("Initiating GridSearchCV (This may take a minute)...")
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='roc_auc', # Optimizing for ROC-AUC, ignoring standard Accuracy
    n_jobs=-1
)

# Fit the grid search ONLY on the training data
grid_search.fit(X_train, y_train)

print(f"Best hyperparameters found: {grid_search.best_params_}")
print(f"Best Cross-Validation ROC-AUC Score: {grid_search.best_score_:.4f}")

# Extract the best model
best_model = grid_search.best_estimator_

Initiating GridSearchCV (This may take a minute)...


### **Task 4:** Final Evaluation on Untouched Test Data
**Goal:** Deploying the best model against the test set and evaluating via Confusion Matrix, Precision, Recall, and ROC-AUC.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Generate predictions on the completely untouched test set
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate final ROC-AUC
final_roc_auc = roc_auc_score(y_test, y_pred_proba)

print("--- CLASSIFICATION REPORT ---")
# This provides Precision, Recall, and F1-Score
print(classification_report(y_test, y_pred))
print(f"Final Test ROC-AUC Score: {final_roc_auc:.4f}\n")

print("--- CONFUSION MATRIX ---")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Default/Fraud (0)', 'Default/Fraud (1)'],
            yticklabels=['Not Default/Fraud (0)', 'Default/Fraud (1)'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix on Test Data')
plt.show()

### **Final Thoughts (What I took away from this project):**


1. **The "Accuracy" Trap:** I learned firsthand that a 95% accurate model is completely useless if it just guesses the majority class and misses the 5% of actual defaults/fraud. Precision, Recall, and ROC-AUC are the only metrics that matter here.

2. **The Zero-Leakage Rule:** Applying SMOTE before splitting the data is a common trap that artificially inflates performance. Implementing imblearn.pipeline to restrict oversampling purely to the training folds was a massive step up in writing production-safe code.

3. **Algorithmic Precision over Guesswork:** Moving beyond simple data cleaning and actually forcing an algorithm (like Random Forest) to learn the structural boundaries of a minority class gave me a much deeper appreciation for how machine learning detects hidden anomalies.